In [2]:
# ============================================================================
# TEST 1: First Call - Cache MISS Expected
# ============================================================================
"""
This should query the LLM since it's the first time we're asking this question.
"""

print("="*70)
print("TEST 1: First Call - Cache MISS Expected")
print("="*70)

response = cached_client.chat_completions_create(
    model=config.vllm.model_name,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.7,
    max_tokens=500
)

print(f"\n📝 Response: {response.choices[0].message.content}")
print(f"\n📊 Status:")
print(f"   Cached: {response.cached}")
print(f"   Similarity Score: {response.similarity_score:.4f}")


TEST 1: First Call - Cache MISS Expected
✗ Cache MISS - calling LLM
✗ Cache MISS - calling LLM

📝 Response: The capital of France is **Paris**.

📊 Status:
   Cached: False
   Similarity Score: 0.0000

📝 Response: The capital of France is **Paris**.

📊 Status:
   Cached: False
   Similarity Score: 0.0000


In [3]:
# ============================================================================
# TEST 2: Exact Same Question - Cache HIT Expected
# ============================================================================
"""
This should hit the cache since we're asking the exact same question.
Similarity score should be 1.0 (perfect match).
"""

print("="*70)
print("TEST 2: Exact Same Question - Cache HIT Expected")
print("="*70)

response = cached_client.chat_completions_create(
    model=config.vllm.model_name,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    temperature=0.7,
    max_tokens=500
)

print(f"\n📝 Response: {response.choices[0].message.content}")
print(f"\n📊 Status:")
print(f"   Cached: {response.cached}")
print(f"   Similarity Score: {response.similarity_score:.4f}")


TEST 2: Exact Same Question - Cache HIT Expected
✓ Cache HIT (similarity: 1.0000)

📝 Response: The capital of France is **Paris**.

📊 Status:
   Cached: True
   Similarity Score: 1.0000


In [4]:
# ============================================================================
# TEST 3: Semantically Similar Question - Cache HIT Expected
# ============================================================================
"""
This asks essentially the same question with different wording.
Should hit the cache if similarity score >= threshold (0.95).
"""

print("="*70)
print("TEST 3: Semantically Similar Question - Cache HIT Expected")
print("="*70)

response = cached_client.chat_completions_create(
    model=config.vllm.model_name,
    messages=[
        {"role": "user", "content": "What's the capital city of France?"}
    ],
    temperature=0.7,
    max_tokens=500
)

print(f"\n📝 Response: {response.choices[0].message.content}")
print(f"\n📊 Status:")
print(f"   Cached: {response.cached}")
print(f"   Similarity Score: {response.similarity_score:.4f}")

if response.cached:
    print(f"\n✅ Semantic cache working! Found similar prompt despite different wording.")
else:
    print(f"\n⚠️  Similarity score too low for cache hit (threshold: {config.cache.similarity_threshold})")


TEST 3: Semantically Similar Question - Cache HIT Expected
✓ Cache HIT (similarity: 0.9855)

📝 Response: The capital of France is **Paris**.

📊 Status:
   Cached: True
   Similarity Score: 0.9855

✅ Semantic cache working! Found similar prompt despite different wording.


In [5]:
# ============================================================================
# TEST 4: Different Question - Cache MISS Expected
# ============================================================================
"""
This is a completely different question, should not hit cache.
"""

print("="*70)
print("TEST 4: Different Question - Cache MISS Expected")
print("="*70)

response = cached_client.chat_completions_create(
    model=config.vllm.model_name,
    messages=[
        {"role": "user", "content": "What is the capital of Germany?"}
    ],
    temperature=0.7,
    max_tokens=500
)

print(f"\n📝 Response: {response.choices[0].message.content}")
print(f"\n📊 Status:")
print(f"   Cached: {response.cached}")
print(f"   Similarity Score: {response.similarity_score:.4f}")


TEST 4: Different Question - Cache MISS Expected
✗ Cache MISS - calling LLM

📝 Response: The capital of Germany is **Berlin**.

📊 Status:
   Cached: False
   Similarity Score: 0.0000

📝 Response: The capital of Germany is **Berlin**.

📊 Status:
   Cached: False
   Similarity Score: 0.0000


In [6]:
# ============================================================================
# Cache Statistics & Performance Analysis
# ============================================================================
"""
View comprehensive statistics about cache performance.
"""

print("="*70)
print("📊 CACHE STATISTICS")
print("="*70)

stats = cached_client.get_stats()

print("\n🎯 Client Statistics:")
print(f"   Total Requests:    {stats['client_stats']['total_requests']}")
print(f"   Cache Hits:        {stats['client_stats']['cache_hits']}")
print(f"   Cache Misses:      {stats['client_stats']['cache_misses']}")
print(f"   Hit Rate:          {stats['client_stats']['hit_rate']:.1%}")

print("\n💾 PostgreSQL Statistics:")
pg_stats = stats['cache_stats']['postgres']
print(f"   Total Entries:     {pg_stats['total_entries']}")
print(f"   Total Accesses:    {pg_stats['total_accesses']}")
print(f"   Avg Accesses:      {pg_stats['avg_accesses_per_entry']:.2f}")
if pg_stats['last_access']:
    print(f"   Last Access:       {pg_stats['last_access']}")

print("\n🔍 Qdrant Statistics:")
qdrant_stats = stats['cache_stats']['qdrant']
print(f"   Vector Points:     {qdrant_stats['points_count']}")
print(f"   Vectors Count:     {qdrant_stats['vectors_count']}")

print("\n⚙️  Configuration:")
print(f"   Similarity Threshold:  {stats['cache_stats']['similarity_threshold']}")
print(f"   Embedding Model:       {stats['cache_stats']['embedding_model']}")
print(f"   Embedding Dimension:   {stats['cache_stats']['embedding_dimension']}")

print("\n" + "="*70)


📊 CACHE STATISTICS

🎯 Client Statistics:
   Total Requests:    4
   Cache Hits:        2
   Cache Misses:      2
   Hit Rate:          50.0%

💾 PostgreSQL Statistics:
   Total Entries:     2
   Total Accesses:    4
   Avg Accesses:      2.00
   Last Access:       2025-10-09T15:30:21.985783

🔍 Qdrant Statistics:
   Vector Points:     2
   Vectors Count:     None

⚙️  Configuration:
   Similarity Threshold:  0.95
   Embedding Model:       nomic-ai/nomic-embed-text-v1.5
   Embedding Dimension:   768



In [7]:
# ============================================================================
# Cache Inspection - View Cached Entries
# ============================================================================
"""
Inspect what's currently in the cache.
"""

print("="*70)
print("🔎 CACHE ENTRIES INSPECTION")
print("="*70)

# Get list of cached entries from PostgreSQL
entries = cache.postgres.list_entries(limit=20)

if not entries:
    print("\n📭 Cache is empty - no entries found.")
else:
    print(f"\n📚 Found {len(entries)} cached entries:\n")
    
    for i, entry in enumerate(entries, 1):
        print(f"{i}. ID: {entry['id']}")
        print(f"   Model: {entry['model']}")
        print(f"   Prompt: {entry['prompt'][:80]}...")
        print(f"   Accesses: {entry['access_count']}")
        print(f"   Created: {entry['created_at']}")
        print(f"   Last Access: {entry['accessed_at']}")
        print()


🔎 CACHE ENTRIES INSPECTION

📚 Found 2 cached entries:

1. ID: 5e0a4b8e-0105-2142-c187-a7c2cd3dea69
   Model: openai/gpt-oss-20b
   Prompt: user: What is the capital of Germany?...
   Accesses: 1
   Created: 2025-10-09 15:30:21.985783
   Last Access: 2025-10-09 15:30:21.985783

2. ID: 60be2138-522f-f326-4f6a-496f2d33071a
   Model: openai/gpt-oss-20b
   Prompt: user: What is the capital of France?...
   Accesses: 3
   Created: 2025-10-09 15:29:53.593405
   Last Access: 2025-10-09 15:30:15.805161



In [ ]:
# ============================================================================
# Cache Management - Clear Cache
# ============================================================================
"""
Clear all cache entries (both PostgreSQL and Qdrant).
Uncomment and run to reset the cache.
"""

# Uncomment the lines below to clear the cache
# print("🗑️  Clearing cache...")
# cached_client.clear_cache()
# print("✅ Cache cleared!")
# 
# # Reset statistics
# cached_client.reset_stats()
# print("✅ Statistics reset!")

print("⚠️  Cache clearing is commented out. Uncomment to use.")


---

## 🎯 Custom Experiments

Use the cells below for your own experiments with the semantic cache.


In [ ]:
# ============================================================================
# Experiment: Your Custom Test
# ============================================================================
"""
Try your own prompts and see how the cache behaves!
"""

custom_prompt = "Explain quantum computing in simple terms."

response = cached_client.chat_completions_create(
    model=config.vllm.model_name,
    messages=[
        {"role": "user", "content": custom_prompt}
    ],
    temperature=0.7,
    max_tokens=500
)

print(f"📝 Response: {response.choices[0].message.content}")
print(f"\n📊 Cached: {response.cached} | Similarity: {response.similarity_score:.4f}")
